# PDF Retrieval with LangChain + Groq (Free & Fast) + Local Embeddings

**What this notebook does (step-by-step):**
1. Loads the PDF
2. Splits it into small pieces (chunks)
3. Converts chunks into numbers (embeddings) using a free local model
4. Stores them in FAISS (fast search engine)
5. When you ask a question → finds the most relevant chunks → sends them + question to Groq Llama 3.3 70B → gives answer

**Why this works well:**
- Local embeddings = no limits on long PDFs
- Groq Llama 3.3 70B = faster + higher limits than Gemini free tier
- 100% free (no billing)

Get free Groq API key: https://console.groq.com/keys

### 1. Install Dependencies

In [1]:
# Run this cell only once
%pip install langchain langchain-classic langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu pypdf sentence-transformers ipywidgets -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Set Your Groq API Key


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
print("✅ Groq API key is set.")

✅ Groq API key is set.


### 3. Load the PDF

PyPDFLoader reads the PDF file and creates one Document object per page. Each document contains the text + metadata (page number).

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# For better testing, use a LONG PDF (50+ pages recommended)

PDF_PATH = "herman-melville-moby-dick.pdf"   # ←←← Make sure the file is in the same folder

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"✅ Loaded {len(pages)} pages.")
print(f"Page 1 preview: {pages[0].page_content[:300]}...")

✅ Loaded 468 pages.
Page 1 preview: HERMAN MELVILLE
MOBYDICK...


### 4. Split into chunks

Long pages are broken into small overlapping pieces (chunks).
chunk_size=600 + chunk_overlap=100 works great for novels because it keeps sentences together while staying inside the LLM’s context window.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # characters per chunk
    chunk_overlap=100,   # overlapping text so no sentence is cut in half
    separators=["\n\n", "\n", " ", ""]   # try to split at natural breaks
)

chunks = splitter.split_documents(pages)

print(f"✅ Split into {len(chunks)} chunks.")
print(f"Chunk 0 preview: {chunks[0].page_content[:200]}...")

✅ Split into 2386 chunks.
Chunk 0 preview: HERMAN MELVILLE
MOBYDICK...


### 5. Create Local Embeddings + Vector Store

Each chunk is converted into a 384-dimensional vector (numbers) using a free local model (all-MiniLM-L6-v2).
FAISS stores these vectors so we can do ultra-fast similarity search later.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    # model_name="sentence-transformers/all-mpnet-base-v2",     # Accurate but slow
    model_name="sentence-transformers/all-MiniLM-L6-v2"         # Fast but less accurate
)

print("Embedding all chunks...")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"✅ Vector store ready with {vectorstore.index.ntotal} vectors.")

Embedding all chunks...
✅ Vector store ready with 2386 vectors.


### 6. Create the Retriever

The retriever is the “search engine” part.
When you give a question, it:

1. Embeds the question
2. Finds the 6 most similar chunks (k=6)

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})   # top-6 chunks

# Quick test to see what it retrieves
test_results = retriever.invoke("What is this document about?")
print("Sanity check - Top retrieved chunks:")
for i, doc in enumerate(test_results):
    print(f"\n--- Chunk {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:250] + "...")

Sanity check - Top retrieved chunks:

--- Chunk 1 (page 77) ---
“All about it, eh—sure you do? —all?”
“Pretty sure.”
With ﬁnger pointed and eye levelled at the Pequod, the beggar-like stranger
stood a moment, as if in a troubled reverie; then starting a little, turned and
said:—“Ye’ve shipped, have ye? Names down...

--- Chunk 2 (page 1) ---
CONTENTS
I. Loomings 
II. e carpet-bag 
III. e Spouter-Inn 
IV . e counterpane 
V . e street 
VI. e chapel 
VII. e pulpit 
VIII. e sermon 
IX. A bosom friend 
X. Nightgown 
XI. Biographical 
XII. Wheelbarrow 
XIII. Nantuck...

--- Chunk 3 (page 347) ---
bulwarks, and with a capacious tub beneath it, into which the minced pieces drop,
fast as the sheets from a rapt orator’s desk. Arrayed in decent black; occupying a
conspicuous pulpit; intent on bible leaves; what a candidate for an archbishop-
rick,...

--- Chunk 4 (page 218) ---
popular work “Goldsmith’s Animated Nature”. In the abridged London edition of
, there ar

### 7. Build the Full RAG Chain (Groq LLM)

This is the complete pipeline:

* Retriever finds chunks
* Prompt puts them together with your question
* Groq Llama 3.3 70B generates the answer

In [7]:
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain          
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain  
from langchain_core.prompts import ChatPromptTemplate

# Fast & powerful Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0               # 0 = factual, no creativity
)

# Prompt
prompt = ChatPromptTemplate.from_template(
    """You are an expert assistant helping answer questions.
Answer the question using ONLY the retrieved context below.
If the context has some relevant information, use it and give the best answer you can.
If the context has almost nothing relevant, say: "The specific part was not retrieved in the top results."

Context:
{context}

Question: {input}

Answer:"""
)

combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("✅ Full RAG pipeline ready! You can now ask questions.")

✅ Full RAG pipeline ready! You can now ask questions.


### 8. Test with Different Questions

In [8]:
test_queries = [
    "Summarize the beginning of the book where the narrator says 'Call me Ishmael'",
    "What is the name of the ship and who is the captain?",
    "Who is Queequeg and what is his relationship with Ishmael?",
    "What is Cetology according to the book?",
    "Describe the White Whale (Moby Dick)",
    "Summarize Chapter 1 (Loomings)",
    "Summarize Chapter 1",
    "Why does Ishmael decide to go to sea?",
    "What happens in the Spouter-Inn?",
    "Why does Ahab hunt the whale?",
    "Quote the first line of the book.",
    "What themes are explored in the novel?",
    "Compare Ahab and Ishmael.",
    "What does the white whale symbolize?"
]

for q in test_queries:
    print(f"\n🔍 Question: {q}")
    response = rag_chain.invoke({"input": q})
    print("🤖 Answer:")
    print(response["answer"][:800] + "..." if len(response["answer"]) > 800 else response["answer"])
    print("-" * 80)


🔍 Question: Summarize the beginning of the book where the narrator says 'Call me Ishmael'
🤖 Answer:
The book begins with the narrator introducing himself as Ishmael. He explains that he has little to no money and is feeling unfulfilled on land, so he decides to sail to see the world and drive off his melancholy. He describes his feelings of gloom and how he often finds himself drawn to morbid things, such as coffin warehouses and funerals. Ishmael then meets Captain Peleg and Bildad, and inquires about shipping out on a whaling voyage, also mentioning that he has a friend who wants to join him. The scene is set in New Bedford, where Ishmael is waiting to embark on his journey, and he is concerned about finding a place to eat and sleep before his departure.
--------------------------------------------------------------------------------

🔍 Question: What is the name of the ship and who is the captain?
🤖 Answer:
The name of the ship is the Pequod, but the captain's name is not explicitl

Why some questions succeed and some fail:
* Questions with specific words from the book (Ishmael, Queequeg, Cetology, Loomings) → very accurate
* Vague chapter questions like “Summarize Chapter 1” without extra context → sometimes weak because the retriever may not pull the exact early pages
* Summarize Chapter 1 (Loomings) -> works because it contain extra context and Exact word from book.
* Descriptive queries (like the first one above) → work excellently

### 9. Interactive Chat Loop (most fun part)

In [9]:
print("🎉 Moby Dick RAG Chatbot is LIVE!")
print("Ask anything about the book. Type 'exit' or 'quit' to stop.\n")

while True:
    query = input("Your question: ")
    if query.lower() in ["exit", "quit", "q"]:
        print("👋 Goodbye!")
        break
    
    response = rag_chain.invoke({"input": query})
    
    print("\n🤖 Answer:")
    print(response["answer"])
    print("\n--- Sources (top 6 chunks) ---")
    for i, doc in enumerate(response["context"]):
        page = doc.metadata.get("page", "?")
        preview = doc.page_content[:150].replace("\n", " ")
        print(f"• Page {page} | {preview}...")
    print("-" * 80)

🎉 Moby Dick RAG Chatbot is LIVE!
Ask anything about the book. Type 'exit' or 'quit' to stop.


🤖 Answer:
The author of this book is Herman Melville, as indicated by "HERMAN MELVILLE" in the context, and the book appears to be "Moby-Dick".

--- Sources (top 6 chunks) ---
• Page 351 | beneath the moon. e sun hides not the ocean, which is the dark side of this earth, and which is two thirds of this earth. So, therefore, that mortal ...
• Page 218 | popular work “Goldsmith’s Animated Nature”. In the abridged London edition of , there are plates of an alleged “whale” and a “narwhale”. I do not ...
• Page 0 | HERMAN MELVILLE MOBYDICK...
• Page 111 | lights of zoology and anatomy. Nevertheless, though of real knowledge there be little, yet of books there are a plenty; and so in some small degree, w...
• Page 268 | ye read it there, Flask? I guess ye did? No; never saw such a book; heard of it, though. But now, tell me, Stubb, do you suppose that that devil you w...
• Page 112 | John Hunt